# 7.1 對話怎麼表示？


一位使用者問「1+1=?」，助手回答「2」。同樣兩段文字，若丟掉誰問誰答，模型只看到接續字句；先把每段的角色與內容分欄，再串成有邊界的序列，才能明確要求它在問題之後回答。

一筆對話是消息清單，每條用role記user或assistant，用content記原文字。沿用前章ByteTokenizer，普通內容每byte一token，BOS、EOS、角色各一專用token。下面把這份問答轉成輸入X與下一項答案Y。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.data import ByteTokenizer, render_chat

messages = [
    {"role": "user", "content": "1+1=?"},
    {"role": "assistant", "content": "2"},
]
tok = ByteTokenizer()
x, y = render_chat(messages, tok)
print("模型輸入X", x.tolist())
print("答案標籤Y", y.tolist())
print("有效目標", y[y != -100].tolist())

`render_chat` 將消息轉成一條序列並建立下一token答案。原始順序為BOS、user、問題bytes、EOS、assistant、答案bytes、EOS。BOS表示開始，EOS表示一段結束；轉換後 x 取原序列除末項的內容；x 第 i 格讀當前 token，y 第 i 格對應原序列的下一項，再把不作為助手答案的位置設成 -100。兩列因此都長10格。輸出X為 `[1,3,57,51,57,69,71,2,4,58]`，其中4是assistant，58是字元2的byte50加8。

Y前八項都是-100，最後為58與2；-100是忽略這個位置的直接答案代價，不是輸入字表ID。`y[y!=-100]` 用真/假條件選出有效項，應輸出 `[58,2]`，要求模型學會回答2以及回答結束。user問題仍在X裡供後面的注意力讀，不需要它們每格都作為loss答案。下一節會看推論開頭，之後再拆監督位置的對齊。

用正確助理示範繼續調整已有模型，叫SFT，監督式微調。文章教「原文接什麼」，問答教「這個問題應答什麼」；模型仍預測下一token，教學材料和監督位置改了。先做資料表示，之後把它接上更新。多輪則按消息順序保留角色，不能靠內容中的標記拼寫猜誰說話。

練習只把問題改成 `"2+1=?"`，先預測角色邊界、長度與有效答案仍58、2，只有問題中的某個byteID改變，再執行核對。答案還固定2，是為了隔離格式變化；若作為正確算術訓練例，應另外將回答改成3，不能讓資料默默帶着舊錯答案。

<details>
<summary>補充：實作約定與原始紀錄</summary>

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
